# TEC Forecasting — LSTM vs Transformer (Single-File Notebook)

Forecasts and evaluates **Dataset 2** for four storm/target days:

| Day | Date | Event |
|---|---|---|
| 33 | 2 May 2024 | G3 storm |
| 37 | 6 May 2024 | G2 storm |
| 40 | 9 May 2024 | Quiet Day |
| 41 | 10 May 2024 | G5 storm |

**Flow**
1. Imports
2. Load datasets and sort them
3. Build matrix, train LSTM & Transformer once, then forecast each target day
   (using the previous day as input) and compute ionospheric delays
4. Plots (generated for every target day) — LSTM loss/validation curve,
   actual vs predicted (LSTM & Transformer), L1+L5 combined ionospheric delay
5. Event-threshold metrics (3 m / 8 m / 15 m) for LSTM & Transformer, L1 & L5,
   for every target day, plus a TEC-level RMSE/MAE summary table

Everything from the original `src/` package is inlined below so the whole
pipeline runs from this one notebook — no project package import needed.


## 1. Imports

**What:** Standard library, numerical/plotting, and PyTorch imports used by
the whole notebook (no project-specific `src.*` imports — everything is
self-contained below).

**How:** `csv`/`re`/`datetime`/`pathlib` handle raw file parsing and paths,
`numpy`/`pandas` handle arrays and result tables, `matplotlib` produces all
figures, and `torch`/`torch.nn`/`DataLoader`/`LambdaLR` build and train the
LSTM and Transformer models.


In [ ]:
import os
import csv
import re
import math
from pathlib import Path
from datetime import datetime
from typing import Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR


### Configuration

**What:** Every path, physical constant, and hyperparameter used downstream,
in one place, plus which dataset/days to run (`DATASET_NO`, `TARGET_DAYS`).

**How:** Paths point at the raw/sorted data and generated-plots folders
(created if missing). `DATASET_DATES`/`get_date_label()` map day numbers to
real calendar dates (with storm labels) for nicer plot titles/filenames.
`get_time_ticks()` builds the shared 6-hourly x-axis ticks used by every
full-day plot. Physical constants (`F_L1`, `F_L2`, `F_L5`, `K_IONO`, …) are
used later to convert TEC → ionospheric delay. `DELAY_EVENT_THRESHOLDS_M`
sets the storm-event thresholds (3 m / 8 m / 15 m) used for POD/CSI/F1/FAR.


In [ ]:
# ---- Data paths ----
BASE_DIR_1   = Path.cwd() / "DataSet" / "DataSet1"
BASE_DIR_2   = Path.cwd() / "DataSet" / "DataSet2"
SORTED_DIR_1 = BASE_DIR_1 / "sortedDataSet"
SORTED_DIR_2 = BASE_DIR_2 / "sortedDataSet"
OUTPUT_DIR_1 = SORTED_DIR_1
OUTPUT_DIR_2 = SORTED_DIR_2

PROJECT_ROOT = Path.cwd()
GENERATED_PLOTS_DIR = PROJECT_ROOT / "generated_plots"
PLOTS_DIR_1 = GENERATED_PLOTS_DIR / "dataset1_plots"
PLOTS_DIR_2 = GENERATED_PLOTS_DIR / "dataset2_plots"

DATASET_DATES = {
    2: {
        33: "02_May_2024_G3_Storm",
        37: "06_May_2024_G2_Storm",
        40: "09_May_2024",
        41: "10_May_2024_G5_Storm",
    },
}

def get_date_label(dataset_id: int, day: int):
    return DATASET_DATES.get(dataset_id, {}).get(day)

#
# Publication-consistent plot styling
#
PLOT_LABEL_FONTSIZE    = 16      
PLOT_LABEL_FONTWEIGHT  = "normal"
PLOT_TICK_FONTSIZE     = 12      
PLOT_TICK_FONTWEIGHT   = "normal"
PLOT_LEGEND_FONTSIZE   = 12
PLOT_LEGEND_FONTWEIGHT = "normal"
PLOT_GRID_ALPHA        = 0.35
PLOT_GRID_LINEWIDTH    = 0.6     
PLOT_DPI               = 300

#
# Plot Figure Sizes
#
PLOT_FIGSIZE_SINGLE  = (14, 6)   # was (9, 12) — full-day series is wide, not tall
PLOT_FIGSIZE_STACKED = (14, 6)
PLOT_FIGSIZE_LOSS    = (9, 5)
PLOT_FIGSIZE_DIURNAL = (11, 5)

#
# Legend placement (shared by every full-day time-series plot)
#
PLOT_LEGEND_ANCHOR = (0.5, 1.02)   
PLOT_LEGEND_LOC     = "lower center"

#
# Line colors — kept here so every plot module (and every model/frequency
# combination) pulls from the same palette instead of redefining its own.
#
COLOR_ACTUAL       = "#2E5EAA"   # steel blue
COLOR_LSTM_PRED    = "#E4572E"   # tomato/orange-red
COLOR_TRANS_PRED   = "#F2A007"   # amber
COLOR_L1_ACTUAL    = "#2E5EAA"
COLOR_L1_PRED      = "#E4572E"
COLOR_L5_ACTUAL    = "#7B2CBF"   # purple
COLOR_L5_PRED      = "#C9184A"   # magenta

def get_time_ticks(minutes_per_day: int = None, interval_hours: int = None):
    if minutes_per_day is None:
        minutes_per_day = MINUTES_PER_DAY
    if interval_hours is None:
        interval_hours = TIME_TICK_INTERVAL_HOURS
    step = interval_hours * 60
    positions = list(range(0, minutes_per_day, step))
    labels = [f"{p // 60:02d}:00" for p in positions]
    positions.append(minutes_per_day - 1)
    labels.append("23:59")
    return positions, labels

# ---- Dataset constants ----
FOLDER_RANGE_1   = (1290, 1690)
FOLDER_RANGE_2   = (910,  1310)
SATELLITES       = range(1, 33)
TOTAL_DAYS       = 41
MINUTES_PER_DAY  = 1440

TRAIN_TARGET_START = 1
TRAIN_TARGET_END   = 30
VAL_TARGET_START   = 31
VAL_TARGET_END     = 41

# ---- Physical constants ----
F_L1     = 1575.42e6
F_L2     = 1227.60e6
F_L5     = 1176.45e6
K_IONO   = 40.308
TECU     = 1e16
R_EARTH  = 6371.0
H_ION    = 350.0

# ---- LSTM hyperparameters ----
LSTM_UNITS_1   = 64
LSTM_UNITS_2   = 32
LSTM_EPOCHS    = 50
LSTM_BATCH     = 4
LSTM_OPTIMIZER = "adam"
LSTM_LOSS      = "mse"
LSTM_SEED      = 42

# ---- Transformer hyperparameters ----
TRANS_D_MODEL       = 64
TRANS_NUM_HEADS     = 4
TRANS_FF_DIM        = 128
TRANS_NUM_LAYERS    = 2
TRANS_DROPOUT       = 0.1
TRANS_EPOCHS        = 50
TRANS_BATCH         = 4
TRANS_LR_INIT       = 1e-4
TRANS_LR_ALPHA      = 1e-6
TRANS_WARMUP_EPOCHS = 10
TRANS_SEED          = 42

# ---- Event thresholds ----
EVENT_THRESHOLD_PERCENTILE = 65.0
DELAY_EVENT_THRESHOLDS_M   = (3.0, 8.0, 15.0)   # storm-event thresholds: 3 m / 8 m / 15 m

# ---- Which dataset / days to run ----
DATASET_NO  = 2                    # 1 or 2
TARGET_DAYS = [33, 37, 40, 41]     # Day 33 (G3 storm), Day 37 (G2 storm), Day 40, Day 41 (G5 storm)


### Model definitions

**What:** The two forecasting architectures, both mapping a full day's
1440-minute TEC sequence to the next day's 1440-minute sequence.

**How:**
- **LSTM** (`LSTMTECModel`): two stacked `nn.LSTM` layers (64 → 32 hidden
  units) followed by a `Linear(32, 1)` applied at every time step —
  sequence-in, sequence-out.
- **Transformer** (`TECTransformerEncoder`): a scalar-to-`d_model` linear
  embedding + fixed sinusoidal positional encoding, fed through a stack of
  Pre-LN `TransformerEncoderLayer`s (self-attention + feed-forward), then
  projected back down to a scalar per minute. Being encoder-only, it
  produces all 1440 predictions in a single forward pass — no
  autoregressive decoding loop.


In [ ]:
# ---- LSTM ----
class LSTMTECModel(nn.Module):
    def __init__(self, input_length=MINUTES_PER_DAY, units_1=LSTM_UNITS_1, units_2=LSTM_UNITS_2):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=1, hidden_size=units_1, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=units_1, hidden_size=units_2, batch_first=True)
        self.fc = nn.Linear(units_2, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        return self.fc(x)

def build_lstm():
    return LSTMTECModel()


# ---- Transformer ----
def positional_encoding(length: int, d_model: int) -> torch.Tensor:
    pe = torch.zeros(length, d_model)
    position = torch.arange(0, length, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0)


class TECTransformerEncoder(nn.Module):
    def __init__(self, seq_len=MINUTES_PER_DAY, d_model=TRANS_D_MODEL, num_heads=TRANS_NUM_HEADS,
                 ff_dim=TRANS_FF_DIM, num_layers=TRANS_NUM_LAYERS, dropout=TRANS_DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(1, d_model)
        self.register_buffer("pos_enc", positional_encoding(seq_len, d_model))
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model))
        self.output_proj = nn.Linear(d_model, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.dropout(self.input_proj(x) + self.pos_enc[:, :x.size(1), :])
        x = self.encoder(x)
        return self.output_proj(x)


def build_cnn_transformer(seq_len=MINUTES_PER_DAY, d_model=TRANS_D_MODEL, num_heads=TRANS_NUM_HEADS,
                           ff_dim=TRANS_FF_DIM, num_layers=TRANS_NUM_LAYERS, dropout=TRANS_DROPOUT):
    return TECTransformerEncoder(seq_len, d_model, num_heads, ff_dim, num_layers, dropout)


### Utility functions
**What:** Physics conversion (TEC → ionospheric delay) and the statistical
metrics used to score every forecast.

**How:**
- `tec_to_iono_delay()` converts TEC (TECU) to a range delay in metres at a
  given frequency: `delay = K_IONO * TECU_const * TEC / frequency²`; an
  optional elevation angle applies a thin-shell obliquity `mapping_function()`
  for slant-path correction (unused here — we compute vertical delay).
- `delay_metrics()` reports RMSE/MAE/bias between predicted and actual delay.
- `rmse()`/`pearson_correlation()` give overall accuracy/agreement.
- `probability_of_detection()`, `critical_success_index()`, `f1_score()`,
  `false_alarm_ratio()` treat "delay ≥ threshold" as a binary storm/event
  and score hits, misses, and false alarms against that threshold — this is
  what powers the Section 5 event-threshold table.


In [ ]:
# ---- Ionospheric delay ----
def mapping_function(elevation_deg):
    elev_rad = np.deg2rad(elevation_deg)
    sin_z_pp = (R_EARTH / (R_EARTH + H_ION)) * np.cos(elev_rad)
    sin_z_pp = np.clip(sin_z_pp, -1.0, 1.0)
    return 1.0 / np.sqrt(1.0 - sin_z_pp ** 2)


def tec_to_iono_delay(tec_tecu, frequency=F_L1, elevation_deg=None):
    alpha = K_IONO * TECU / (frequency ** 2)
    delay_v = alpha * tec_tecu
    if elevation_deg is not None:
        return delay_v * mapping_function(elevation_deg)
    return delay_v


def compute_all_delays(actual, lstm_pred, trans_pred, frequency=F_L1):
    return (
        tec_to_iono_delay(actual, frequency=frequency),
        tec_to_iono_delay(lstm_pred, frequency=frequency),
        tec_to_iono_delay(trans_pred, frequency=frequency),
    )


def delay_metrics(name, predicted, actual, verbose=True):
    err = predicted - actual
    rmse_v = float(np.sqrt(np.mean(err ** 2)))
    mae_v = float(np.mean(np.abs(err)))
    bias_v = float(np.mean(err))
    if verbose:
        print(f"{name:<15}  RMSE = {rmse_v:.5f} m   MAE = {mae_v:.5f} m   Bias = {bias_v:+.5f} m")
    return dict(name=name, rmse=rmse_v, mae=mae_v, bias=bias_v)


# ---- Metrics ----
def rmse(predicted, actual):
    predicted, actual = np.asarray(predicted, float), np.asarray(actual, float)
    return float(np.sqrt(np.mean((predicted - actual) ** 2)))


def pearson_correlation(predicted, actual):
    predicted, actual = np.asarray(predicted, float), np.asarray(actual, float)
    if np.std(predicted) == 0 or np.std(actual) == 0:
        return float("nan")
    return float(np.corrcoef(predicted.ravel(), actual.ravel())[0, 1])


def probability_of_detection(predicted, actual, threshold):
    predicted, actual = np.asarray(predicted, float), np.asarray(actual, float)
    predicted_event, actual_event = predicted >= threshold, actual >= threshold
    hits = np.sum(predicted_event & actual_event)
    misses = np.sum(~predicted_event & actual_event)
    return float(hits / (hits + misses)) if (hits + misses) else float("nan")


def critical_success_index(predicted, actual, threshold):
    predicted, actual = np.asarray(predicted, float), np.asarray(actual, float)
    predicted_event, actual_event = predicted >= threshold, actual >= threshold
    hits = np.sum(predicted_event & actual_event)
    false_alarms = np.sum(predicted_event & ~actual_event)
    misses = np.sum(~predicted_event & actual_event)
    denom = hits + false_alarms + misses
    return float(hits / denom) if denom else float("nan")


def f1_score(predicted, actual, threshold):
    predicted, actual = np.asarray(predicted, float), np.asarray(actual, float)
    predicted_event, actual_event = predicted >= threshold, actual >= threshold
    hits = np.sum(predicted_event & actual_event)
    false_alarms = np.sum(predicted_event & ~actual_event)
    misses = np.sum(~predicted_event & actual_event)
    denom = 2 * hits + false_alarms + misses
    return float(2 * hits / denom) if denom else float("nan")


def false_alarm_ratio(predicted, actual, threshold):
    predicted, actual = np.asarray(predicted, float), np.asarray(actual, float)
    predicted_event, actual_event = predicted >= threshold, actual >= threshold
    hits = np.sum(predicted_event & actual_event)
    false_alarms = np.sum(predicted_event & ~actual_event)
    denom = hits + false_alarms
    return float(false_alarms / denom) if denom else float("nan")


## 2. Load datasets and sort them

**What:** Function definitions for turning raw per-satellite CSVs into a
clean per-day TEC matrix ready for training. (Executed in the cells that
follow.)

**How:**
- `discover_source_folders()` finds the `iisc*_TECU` day-folders in range
  for the chosen dataset.
- `aggregate_minute_max()` reads all 32 satellites' CSVs for one day-folder
  and keeps, per UTC minute, the **maximum** TEC value seen across
  satellites — a simple way to collapse multi-satellite noise into one
  signal per minute.
- `write_sorted_csv()` writes that as a tidy `Time(UTC), TEC(TECU)` CSV.
- `build_sorted_dataset()` runs this for every day-folder.
- `load_day_vector()` reads one sorted CSV into a length-1440 array,
  linearly interpolating any missing minutes.
- `build_daily_matrix()` stacks the first 41 sorted days into a
  `(41, 1440)` matrix.
- `build_splits()` turns that into supervised `(X=day N, y=day N+1)` pairs
  and splits them into train (target days 1–30) / validation (31–41).
- `normalise()` z-score normalises using **training-set** mean/std only,
  and `prepare_dataset()` chains all of the above into one call.


In [ ]:
# ---- Sorting raw per-satellite CSVs into minute-wise max TEC ----
def discover_source_folders(data_set_no: int = 1):
    if data_set_no == 1:
        lo, hi, base_dir = *FOLDER_RANGE_1, BASE_DIR_1
    elif data_set_no == 2:
        lo, hi, base_dir = *FOLDER_RANGE_2, BASE_DIR_2
    else:
        raise ValueError(f"data_set_no must be 1 or 2, got {data_set_no}")

    pattern = re.compile(r"^iisc(\d{4})_TECU$")
    folders = []
    for p in base_dir.iterdir():
        if not p.is_dir():
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        n = int(m.group(1))
        if lo <= n <= hi:
            folders.append((n, p))
    folders.sort(key=lambda x: x[0])
    return folders


def aggregate_minute_max(folder_path: Path) -> dict:
    minute_max = {}
    for sat_idx in SATELLITES:
        csv_path = folder_path / f"G{sat_idx:02d}_TEC.csv"
        if not csv_path.exists():
            continue
        with csv_path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                time_text = (row.get("Time (UTC)") or "").strip()
                tec_text = (row.get("TEC (TECU)") or "").strip()
                if not time_text or not tec_text:
                    continue
                try:
                    dt = datetime.strptime(time_text, "%Y-%m-%d %H:%M:%S")
                    tec = float(tec_text)
                except ValueError:
                    continue
                minute_dt = dt.replace(second=0)
                prev = minute_max.get(minute_dt)
                if prev is None or tec > prev:
                    minute_max[minute_dt] = tec
    return minute_max


def write_sorted_csv(minute_max: dict, out_path: Path) -> None:
    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Time(UTC)", "TEC(TECU)"])
        for minute_dt in sorted(minute_max.keys()):
            writer.writerow([minute_dt.strftime("%d-%m-%y %H:%M"), f"{minute_max[minute_dt]:.6f}"])


def build_sorted_dataset(data_set_no: int = 1) -> int:
    output_dir = OUTPUT_DIR_1 if data_set_no == 1 else OUTPUT_DIR_2
    output_dir.mkdir(parents=True, exist_ok=True)
    folders = discover_source_folders(data_set_no)
    written = 0
    for _, folder_path in folders:
        minute_max = aggregate_minute_max(folder_path)
        out_path = output_dir / f"{folder_path.name}_sorted.csv"
        write_sorted_csv(minute_max, out_path)
        written += 1
    return written


# ---- Build (days x minutes) matrix, splits, normalisation ----
MINUTE_LABELS = [f"{h:02d}:{m:02d}" for h in range(24) for m in range(60)]
MINUTE_INDEX = {t: i for i, t in enumerate(MINUTE_LABELS)}


def load_day_vector(csv_path: Path) -> np.ndarray:
    vec = np.full(MINUTES_PER_DAY, np.nan, dtype=np.float32)
    with csv_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = (row.get("Time(UTC)") or "").strip()
            v = (row.get("TEC(TECU)") or "").strip()
            if not t or not v:
                continue
            try:
                dt = datetime.strptime(t, "%d-%m-%y %H:%M")
                tec = float(v)
            except ValueError:
                continue
            idx = MINUTE_INDEX.get(dt.strftime("%H:%M"))
            if idx is not None:
                vec[idx] = tec
    if np.isnan(vec).any():
        x = np.arange(len(vec))
        valid = ~np.isnan(vec)
        if valid.sum() == 0:
            raise ValueError(f"No usable TEC data in: {csv_path.name}")
        vec = np.interp(x, x[valid], vec[valid]).astype(np.float32)
    return vec


def build_daily_matrix(data_set_no: int = 1):
    sorted_dir = SORTED_DIR_1 if data_set_no == 1 else SORTED_DIR_2
    all_csv = sorted(sorted_dir.glob("*_sorted.csv"))
    if len(all_csv) < TOTAL_DAYS:
        raise ValueError(f"Need at least {TOTAL_DAYS} sorted CSV files, found {len(all_csv)}")
    daily_files = all_csv[:TOTAL_DAYS]
    daily_matrix = np.stack([load_day_vector(p) for p in daily_files], axis=0)
    return daily_matrix, daily_files


def build_splits(daily_matrix: np.ndarray):
    X_all = daily_matrix[:-1]
    y_all = daily_matrix[1:]
    target_day_num = np.arange(2, TOTAL_DAYS + 1)
    train_mask = (target_day_num >= TRAIN_TARGET_START) & (target_day_num <= TRAIN_TARGET_END)
    val_mask = (target_day_num >= VAL_TARGET_START) & (target_day_num <= VAL_TARGET_END)
    return X_all[train_mask], y_all[train_mask], X_all[val_mask], y_all[val_mask]


def normalise(X_train_raw, y_train_raw, X_val_raw, y_val_raw):
    x_mean, x_std = float(X_train_raw.mean()), float(X_train_raw.std()) + 1e-8
    y_mean, y_std = float(y_train_raw.mean()), float(y_train_raw.std()) + 1e-8
    X_train = ((X_train_raw - x_mean) / x_std)[..., np.newaxis]
    y_train = ((y_train_raw - y_mean) / y_std)[..., np.newaxis]
    X_val = ((X_val_raw - x_mean) / x_std)[..., np.newaxis]
    y_val = ((y_val_raw - y_mean) / y_std)[..., np.newaxis]
    stats = dict(x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std)
    return X_train, y_train, X_val, y_val, stats


def prepare_dataset(data_set_no: int = 1):
    daily_matrix, daily_files = build_daily_matrix(data_set_no)
    X_train_r, y_train_r, X_val_r, y_val_r = build_splits(daily_matrix)
    X_train, y_train, X_val, y_val, stats = normalise(X_train_r, y_train_r, X_val_r, y_val_r)
    return X_train, y_train, X_val, y_val, stats, daily_matrix, daily_files


### Run sorting for the selected dataset

**What:** Executes `build_sorted_dataset()` on the raw per-satellite TEC
folders (`iisc*_TECU`) for `DATASET_NO`.

**How:** For every day-folder in range, it reads all 32 satellite CSVs,
keeps the maximum TEC value observed in each UTC minute (across satellites),
and writes one `*_sorted.csv` per day into `sortedDataSet/`. This collapses
noisy multi-satellite readings into a single clean 1440-row time series
per day, which is what the model actually trains on.


In [ ]:
# ---- Run sorting + loading for the selected dataset ----
BASE_DIR   = BASE_DIR_1   if DATASET_NO == 1 else BASE_DIR_2
OUTPUT_DIR = OUTPUT_DIR_1 if DATASET_NO == 1 else OUTPUT_DIR_2
PLOTS_DIR  = PLOTS_DIR_1  if DATASET_NO == 1 else PLOTS_DIR_2

print(f"Base directory   : {BASE_DIR}")
print(f"Output directory : {OUTPUT_DIR}")

written_files = build_sorted_dataset(DATASET_NO)
print(f"TEC data sorting completed — Dataset {DATASET_NO}.")
print(f"Number of output CSV files written: {written_files}")
print(f"Saved sorted CSV files to: {OUTPUT_DIR}")


### Build the daily matrix and normalised train/val splits

**What:** Calls `prepare_dataset()`, which loads all 41 sorted CSVs into a
`(41, 1440)` matrix (one row per day, one column per minute), builds
supervised pairs (`X = day N`, `y = day N+1`), splits them into train
(target days 1–30) and validation (target days 31–41), and z-score
normalises everything using **training-set statistics only** (so the
validation/test days never leak into the normalisation).

**How:** Each `X`/`y` array ends up shaped `(N, 1440, 1)` — a trailing
channel dimension the LSTM/Transformer expect. `stats` stores the mean/std
used, so predictions can later be de-normalised back into TECU.


In [ ]:
X_train, y_train, X_val, y_val, stats, daily_matrix, daily_files = prepare_dataset(DATASET_NO)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"Norm stats : {stats}")


## 3. Train LSTM & Transformer, forecast, and compute ionospheric delays

In [ ]:
# ---- Training loops (inlined lstm_training.py / transformer_training.py) ----
def train_lstm(X_train, y_train, X_val, y_val, epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH):
    torch.manual_seed(LSTM_SEED)
    np.random.seed(LSTM_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    X_v  = torch.tensor(X_val,   dtype=torch.float32)
    y_v  = torch.tensor(y_val,   dtype=torch.float32)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    model = build_lstm().to(device)
    optimizer = torch.optim.Adam(model.parameters())
    criterion = nn.MSELoss()
    history = {"loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_v.to(device)), y_v.to(device)).item()

        history["loss"].append(epoch_loss / len(loader))
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch+1}/{epochs} — loss: {history['loss'][-1]:.6f} — val_loss: {val_loss:.6f}")

    return model, history


def get_warmup_cosine_schedule(optimizer, warmup_steps, total_steps, peak_lr, alpha):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        floor = alpha / peak_lr
        return floor + (1.0 - floor) * cosine
    return LambdaLR(optimizer, lr_lambda)


def train_transformer(X_train, y_train, X_val, y_val, epochs=TRANS_EPOCHS, batch_size=TRANS_BATCH):
    torch.manual_seed(TRANS_SEED)
    np.random.seed(TRANS_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Transformer] Training on: {device}")

    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    X_v  = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_v  = torch.tensor(y_val,   dtype=torch.float32).to(device)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    model = build_cnn_transformer().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRANS_LR_INIT)
    criterion = nn.MSELoss()
    print(model)

    steps_per_epoch = max(1, len(X_train) // batch_size)
    total_steps = epochs * steps_per_epoch
    warmup_steps = TRANS_WARMUP_EPOCHS * steps_per_epoch

    scheduler = get_warmup_cosine_schedule(optimizer, warmup_steps, total_steps, TRANS_LR_INIT, TRANS_LR_ALPHA)

    history = {"loss": [], "val_loss": []}
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_v)
            val_loss = criterion(val_pred, y_v).item()

        avg_train = epoch_loss / len(loader)
        history["loss"].append(avg_train)
        history["val_loss"].append(val_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:3d}/{epochs} | loss: {avg_train:.6f} | val_loss: {val_loss:.6f} | lr: {current_lr:.2e}")

    return model, history


### Train the LSTM

**What:** Calls `train_lstm()` on the normalised train/val tensors.

**How:** A 2-layer stacked LSTM (`units_1=64` → `units_2=32`) reads the
1440-minute input sequence and outputs a 1440-minute prediction at every
time step (sequence-to-sequence, via `TimeDistributed`-style `Linear` on
each step). It trains with Adam + MSE loss for `LSTM_EPOCHS` epochs,
printing train/val loss every epoch, and returns the trained model plus a
`history` dict of per-epoch losses (used for the loss-curve plot later).


In [ ]:
# ---- Train both models ----
lstm_model, lstm_history = train_lstm(X_train, y_train, X_val, y_val)


### Train the Transformer

**What:** Calls `train_transformer()` on the same data.

**How:** An encoder-only Transformer (`d_model=64`, 4 heads, 2 layers)
projects each scalar TEC value to `d_model`, adds sinusoidal positional
encoding, runs it through pre-LN self-attention layers, and projects back
to a scalar per minute — one single forward pass predicts the whole next
day (no autoregressive decoding). Training uses Adam with a linear-warmup
+ cosine-decay learning-rate schedule and gradient clipping for stability,
again returning the trained model and a `history` dict.


In [ ]:
transformer_model, transformer_history = train_transformer(X_train, y_train, X_val, y_val)


### Forecast every target day (33, 37, 40, 41)

**What:** For each day in `TARGET_DAYS`, forecasts that day's TEC using
**both** trained models and computes the matching L1 ionospheric delay.

**How:** Each forecast is one-step-ahead: day `N`'s actual (raw, then
normalised with the *training* mean/std) is fed in, and the model outputs
day `N+1`'s normalised prediction, which is de-normalised back to TECU
using the training `y_mean`/`y_std`. This is repeated once per model per
day. Everything (actual, both predictions, both dates, and their vertical
L1 delay via `tec_to_iono_delay`) is cached in the `results` dict, keyed
by day number, so every later plot/metric cell just reads from `results`
instead of recomputing.


In [ ]:
# ---- Forecast every day in TARGET_DAYS using the previous day's data as input ----
x_mean, x_std = stats["x_mean"], stats["x_std"]
y_mean, y_std = stats["y_mean"], stats["y_std"]

device = next(lstm_model.parameters()).device
lstm_model.eval()
transformer_model.eval()

# results[day] = dict(actual=..., lstm_pred=..., trans_pred=..., date_label=...,
#                      iono_actual=..., iono_lstm=..., iono_trans=...)
results = {}

for day in TARGET_DAYS:
    input_day_raw = daily_matrix[day - 2]   # day N-1 -> predicts day N
    actual_day    = daily_matrix[day - 1]

    input_norm = ((input_day_raw - x_mean) / x_std)[np.newaxis, ..., np.newaxis]
    src = torch.tensor(input_norm, dtype=torch.float32).to(device)

    with torch.no_grad():
        lstm_pred_norm  = lstm_model(src).squeeze().cpu().numpy()
        trans_pred_norm = transformer_model(src).squeeze().cpu().numpy()

    lstm_pred_day  = lstm_pred_norm  * y_std + y_mean
    trans_pred_day = trans_pred_norm * y_std + y_mean

    date_label_day = get_date_label(DATASET_NO, day)

    results[day] = dict(
        actual=actual_day,
        lstm_pred=lstm_pred_day,
        trans_pred=trans_pred_day,
        date_label=date_label_day,
        iono_actual=tec_to_iono_delay(actual_day),
        iono_lstm=tec_to_iono_delay(lstm_pred_day),
        iono_trans=tec_to_iono_delay(trans_pred_day),
    )

    print(f"Forecast day: {day}  ({date_label_day or 'date unknown'})  — done")


## 4. Plots — LSTM loss/validation, actual vs predicted, L1+L5 combined ionospheric delay

### Shared plot styling helpers

**What:** Small utilities reused by every plot below: consistent time-of-day
x-axis ticks (`_TICK_POS`/`_TICK_LABELS`, every 6 hours), frequency-name
lookup, day/date label formatting, filename building, and two shared
helpers — `_style_timeseries_ax()` (axis labels/ticks/grid) and
`_legend_above()` (legend placement) — that both read every visual setting
(fonts, sizes, colors, grid, legend anchor) straight from the config cell,
so changing one constant there updates every plot consistently.


In [ ]:
# ---- Shared plot style helpers ----
_TICK_POS, _TICK_LABELS = get_time_ticks()
_FREQ_NAMES = {F_L1: "L1", F_L2: "L2", F_L5: "L5"}

def _freq_name(freq):
    return _FREQ_NAMES.get(freq, str(freq))

def _day_or_date(target_day, date_label=None):
    return date_label if date_label else f"day{target_day}"

def _make_filename(*parts):
    clean = [str(p) for p in parts if p not in (None, "")]
    return "_".join(clean) + ".png"

def _style_timeseries_ax(ax, ylabel):
    """Shared axis styling for every full-day time-series plot — reads
    fonts/grid settings from config so every plot stays visually consistent."""
    ax.set_xlabel("Time of Day (UTC)", fontsize=PLOT_LABEL_FONTSIZE, fontweight=PLOT_LABEL_FONTWEIGHT)
    ax.set_ylabel(ylabel, fontsize=PLOT_LABEL_FONTSIZE, fontweight=PLOT_LABEL_FONTWEIGHT)
    ax.set_xticks(_TICK_POS)
    ax.set_xticklabels(_TICK_LABELS, fontsize=PLOT_TICK_FONTSIZE, fontweight=PLOT_TICK_FONTWEIGHT)
    ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
    for t in ax.get_yticklabels():
        t.set_fontweight(PLOT_TICK_FONTWEIGHT)
    ax.grid(True, alpha=PLOT_GRID_ALPHA, linewidth=PLOT_GRID_LINEWIDTH)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

def _legend_above(ax, ncol=2):
    """Legend on a fixed anchor above the axes (config-driven position) so it
    never overlaps the data, regardless of where the curve peaks."""
    ax.legend(
        loc=PLOT_LEGEND_LOC,
        bbox_to_anchor=PLOT_LEGEND_ANCHOR,
        ncol=ncol,
        frameon=False,
        prop={"weight": PLOT_LEGEND_FONTWEIGHT, "size": PLOT_LEGEND_FONTSIZE},
        handlelength=2.2,
        columnspacing=1.5,
    )


### LSTM training/validation loss curve

**What:** Plots `lstm_history["loss"]` vs `lstm_history["val_loss"]` across
epochs and saves it as `loss_LSTM_dataset2.png`.

**How:** `plot_lstm_loss()` draws both curves on one figure (training solid
circles, validation dashed squares), thins the x-tick labels to every 10th
epoch to avoid clutter, applies the shared publication styling, saves the
figure to `PLOTS_DIR`, then prints the final train/val loss values. A
validation curve that flattens or rises while training loss keeps falling
would indicate overfitting.


In [ ]:
# ---- LSTM loss / validation curve ----
def plot_lstm_loss(history, dataset_label=None, output_dir="plots", save=True):
    epochs_range = range(1, len(history["loss"]) + 1)
    tick_positions = [e for e in epochs_range if e == 1 or e % 10 == 0]

    plt.figure(figsize=PLOT_FIGSIZE_LOSS)
    plt.plot(epochs_range, history["loss"], marker="o", linewidth=1.5, label="Training Loss")
    plt.plot(epochs_range, history["val_loss"], marker="s", linewidth=1.5, linestyle="--", label="Validation Loss")
    plt.xlabel("Epoch", fontsize=PLOT_LABEL_FONTSIZE, fontweight=PLOT_LABEL_FONTWEIGHT)
    plt.ylabel("MSE Loss", fontsize=PLOT_LABEL_FONTSIZE, fontweight=PLOT_LABEL_FONTWEIGHT)
    plt.xticks(ticks=tick_positions, labels=[str(e) for e in tick_positions],
               fontsize=PLOT_TICK_FONTSIZE, fontweight=PLOT_TICK_FONTWEIGHT)
    plt.yticks(fontsize=PLOT_TICK_FONTSIZE, fontweight=PLOT_TICK_FONTWEIGHT)
    plt.legend(prop={"weight": PLOT_LEGEND_FONTWEIGHT, "size": PLOT_LEGEND_FONTSIZE})
    plt.grid(True, alpha=PLOT_GRID_ALPHA)
    plt.tight_layout()

    if save:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, _make_filename("loss_LSTM", dataset_label))
        plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"Saved: {path}")

    plt.show()
    print(f"Final train loss : {history['loss'][-1]:.6f}")
    print(f"Final val   loss : {history['val_loss'][-1]:.6f}")


plot_lstm_loss(lstm_history, dataset_label=f"dataset{DATASET_NO}", output_dir=str(PLOTS_DIR))


### Actual vs predicted TEC — every forecast day, both models

**What:** For each of the 4 target days, plots the actual full-day TEC
curve against the LSTM prediction and, separately, against the Transformer
prediction, then prints/stores RMSE and MAE (in TECU) for each.

**How:** `plot_lstm_prediction()`/`plot_transformer_prediction()` use
`PLOT_FIGSIZE_SINGLE` (wide/landscape) and draw the actual series
(`COLOR_ACTUAL`) against the predicted series (`COLOR_LSTM_PRED` /
`COLOR_TRANS_PRED`, dashed) on the same time-of-day axis. The legend sits
on a fixed anchor above the axes (`PLOT_LEGEND_ANCHOR`/`PLOT_LEGEND_LOC`)
so it never overlaps the curve, however the diurnal shape peaks. Each
figure is saved with a filename encoding model, date, and dataset, and
`(rmse, mae)` is returned and collected into `tec_metrics[day]` for the
Section 5 summary table.


In [ ]:
# ---- Actual vs predicted TEC plots ----
def plot_lstm_prediction(actual, lstm_pred, target_day=41, dataset_label=None, date_label=None,
                          output_dir="plots", save=True):
    minutes = np.arange(len(actual))
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_SINGLE)

    ax.plot(minutes, actual, color=COLOR_ACTUAL, linewidth=1.8, label="Actual", zorder=3)
    ax.plot(minutes, lstm_pred, color=COLOR_LSTM_PRED, linewidth=1.6,
            linestyle="--", label="LSTM Predicted", zorder=3)
    ax.fill_between(minutes, actual, lstm_pred, color=COLOR_LSTM_PRED, alpha=0.12, zorder=1)

    ax.set_xlim(0, len(minutes) - 1)
    _style_timeseries_ax(ax, "TEC (TECU)")
    _legend_above(ax, ncol=2)
    fig.tight_layout(rect=[0, 0, 1, 0.94])

    if save:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, _make_filename("actual_vs_predicted_LSTM", _day_or_date(target_day, date_label), dataset_label))
        fig.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"Saved: {path}")

    plt.show()
    rmse_v = float(np.sqrt(np.mean((lstm_pred - actual) ** 2)))
    mae_v = float(np.mean(np.abs(lstm_pred - actual)))
    print(f"LSTM Day-{target_day} RMSE : {rmse_v:.4f} TECU")
    print(f"LSTM Day-{target_day} MAE  : {mae_v:.4f} TECU")
    return rmse_v, mae_v


def plot_transformer_prediction(actual, trans_pred, target_day=41, dataset_label=None, date_label=None,
                                 output_dir="plots", save=True):
    minutes = np.arange(len(actual))
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_SINGLE)

    ax.plot(minutes, actual, color=COLOR_ACTUAL, linewidth=1.8, label="Actual", zorder=3)
    ax.plot(minutes, trans_pred, color=COLOR_TRANS_PRED, linewidth=1.6,
            linestyle="--", label="Transformer Predicted", zorder=3)
    ax.fill_between(minutes, actual, trans_pred, color=COLOR_TRANS_PRED, alpha=0.15, zorder=1)

    ax.set_xlim(0, len(minutes) - 1)
    _style_timeseries_ax(ax, "TEC (TECU)")
    _legend_above(ax, ncol=2)
    fig.tight_layout(rect=[0, 0, 1, 0.94])

    if save:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, _make_filename("actual_vs_predicted_transformer", _day_or_date(target_day, date_label), dataset_label))
        fig.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"Saved: {path}")

    plt.show()
    rmse_v = float(np.sqrt(np.mean((trans_pred - actual) ** 2)))
    mae_v = float(np.mean(np.abs(trans_pred - actual)))
    print(f"Transformer Day-{target_day} RMSE : {rmse_v:.4f} TECU")
    print(f"Transformer Day-{target_day} MAE  : {mae_v:.4f} TECU")
    return rmse_v, mae_v


tec_metrics = {}   # tec_metrics[day] = dict(lstm_rmse=, lstm_mae=, trans_rmse=, trans_mae=)

for day in TARGET_DAYS:
    r = results[day]
    print(f"\n{'='*70}\nDay {day}  ({r['date_label'] or 'date unknown'})\n{'='*70}")

    lstm_rmse_d, lstm_mae_d = plot_lstm_prediction(
        r["actual"], r["lstm_pred"], target_day=day, dataset_label=f"dataset{DATASET_NO}",
        date_label=r["date_label"], output_dir=str(PLOTS_DIR),
    )
    trans_rmse_d, trans_mae_d = plot_transformer_prediction(
        r["actual"], r["trans_pred"], target_day=day, dataset_label=f"dataset{DATASET_NO}",
        date_label=r["date_label"], output_dir=str(PLOTS_DIR),
    )

    tec_metrics[day] = dict(
        lstm_rmse=lstm_rmse_d, lstm_mae=lstm_mae_d,
        trans_rmse=trans_rmse_d, trans_mae=trans_mae_d,
    )


### Combined L1 + L5 ionospheric delay — every forecast day, both models

**What:** For each target day, overlays actual vs predicted ionospheric
delay for **both** L1 and L5 on one figure (per model), so the two
frequency bands can be compared directly.

**How:** `_plot_combined_l1_l5_delay()` converts TEC → delay in metres for
each frequency via `tec_to_iono_delay()`, plots actual (solid) vs predicted
(dashed) for L1 (`COLOR_L1_*`, blue) and L5 (`COLOR_L5_*`, purple/magenta)
on shared axes with a shaded error band, prints per-frequency RMSE/MAE, and
saves one figure per model per day. The legend (4 entries, 2x2 grid) uses
the same fixed above-axes anchor as the prediction plots so it never sits
on top of the storm-day peaks. `plot_lstm_delay_l1_l5_combined` and
`plot_transformer_delay_l1_l5_combined` are thin wrappers fixing the
prediction source and color.


In [ ]:
# ---- Combined L1 + L5 ionospheric delay plots ----
def _plot_combined_l1_l5_delay(actual, pred, pred_label, pred_color, target_day, dataset_label,
                                date_label, output_dir, save, filename_prefix):
    minutes = np.arange(len(actual))
    freq_styles = [
        (F_L1, "L1", COLOR_L1_ACTUAL, COLOR_L1_PRED, "-", "--"),
        (F_L5, "L5", COLOR_L5_ACTUAL, COLOR_L5_PRED, "-", "--"),
    ]
    fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_SINGLE)
    for freq, fname, actual_color, pred_color_f, actual_ls, pred_ls in freq_styles:
        iono_actual = tec_to_iono_delay(actual, frequency=freq)
        iono_pred = tec_to_iono_delay(pred, frequency=freq)
        ax.plot(minutes, iono_actual, color=actual_color, linewidth=1.6, linestyle=actual_ls, label=f"Actual ({fname})", zorder=3)
        ax.plot(minutes, iono_pred, color=pred_color_f, linewidth=1.4, linestyle=pred_ls, label=f"{pred_label} ({fname})", zorder=3)
        ax.fill_between(minutes, iono_actual, iono_pred, alpha=0.10, color=pred_color_f, zorder=1)
        rmse_v = np.sqrt(np.mean((iono_pred - iono_actual) ** 2))
        mae_v = np.mean(np.abs(iono_pred - iono_actual))
        print(f"{pred_label} {fname} — RMSE: {rmse_v:.5f} m   MAE: {mae_v:.5f} m")

    ax.set_xlim(0, len(minutes) - 1)
    _style_timeseries_ax(ax, "Iono. Delay (m)")
    _legend_above(ax, ncol=2)   # 4 legend entries -> 2x2 grid above the axes
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    if save:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, _make_filename(filename_prefix, "L1_L5_combined", _day_or_date(target_day, date_label), dataset_label))
        fig.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
        print(f"Saved: {path}")

    plt.show()


def plot_lstm_delay_l1_l5_combined(actual, lstm_pred, target_day=41, dataset_label=None,
                                    date_label=None, output_dir="plots", save=True):
    _plot_combined_l1_l5_delay(actual, lstm_pred, "LSTM Predicted", COLOR_LSTM_PRED,
                                target_day, dataset_label, date_label, output_dir, save,
                                "ionospheric_delay_LSTM")


def plot_transformer_delay_l1_l5_combined(actual, trans_pred, target_day=41, dataset_label=None,
                                           date_label=None, output_dir="plots", save=True):
    _plot_combined_l1_l5_delay(actual, trans_pred, "Transformer Predicted", COLOR_TRANS_PRED,
                                target_day, dataset_label, date_label, output_dir, save,
                                "ionospheric_delay_transformer")


for day in TARGET_DAYS:
    r = results[day]
    print(f"\n{'='*70}\nDay {day}  ({r['date_label'] or 'date unknown'}) — L1+L5 combined delay\n{'='*70}")

    plot_lstm_delay_l1_l5_combined(
        r["actual"], r["lstm_pred"], target_day=day, dataset_label=f"dataset{DATASET_NO}",
        date_label=r["date_label"], output_dir=str(PLOTS_DIR),
    )
    plot_transformer_delay_l1_l5_combined(
        r["actual"], r["trans_pred"], target_day=day, dataset_label=f"dataset{DATASET_NO}",
        date_label=r["date_label"], output_dir=str(PLOTS_DIR),
    )


## 5. Event-threshold metrics — LSTM & Transformer, L1 & L5, all forecast days

For each forecast day in `TARGET_DAYS` and each ionospheric-delay threshold in
`DELAY_EVENT_THRESHOLDS_M` (3 m, 8 m, 15 m — edit this tuple in the config cell
if you want 3 / 5 / 15 m exactly), compute POD, CSI, F1 and FAR for both models
at both L1 and L5, plus overall RMSE/Correlation and TEC-level RMSE/MAE.


In [ ]:
freqs = {"L1": F_L1, "L5": F_L5}
event_rows = []

for day in TARGET_DAYS:
    r = results[day]
    actual_day, lstm_pred_day, trans_pred_day = r["actual"], r["lstm_pred"], r["trans_pred"]

    for freq_label, freq_hz in freqs.items():
        iono_actual_f, iono_lstm_f, iono_trans_f = compute_all_delays(
            actual_day, lstm_pred_day, trans_pred_day, frequency=freq_hz
        )

        for model_name, pred_delay in (("LSTM", iono_lstm_f), ("Transformer", iono_trans_f)):
            overall_rmse = rmse(pred_delay, iono_actual_f)
            overall_corr = pearson_correlation(pred_delay, iono_actual_f)

            for thr in DELAY_EVENT_THRESHOLDS_M:
                pod = probability_of_detection(pred_delay, iono_actual_f, threshold=thr)
                csi = critical_success_index(pred_delay, iono_actual_f, threshold=thr)
                f1  = f1_score(pred_delay, iono_actual_f, threshold=thr)
                far = false_alarm_ratio(pred_delay, iono_actual_f, threshold=thr)

                event_rows.append({
                    "Day": day,
                    "Date": r["date_label"] or f"day{day}",
                    "Frequency": freq_label,
                    "Model": model_name,
                    "Threshold (m)": thr,
                    "Delay RMSE (m)": overall_rmse,
                    "Delay Correlation": overall_corr,
                    "POD": pod,
                    "CSI": csi,
                    "F1 Score": f1,
                    "FAR": far,
                })

event_metrics_table = pd.DataFrame(event_rows).round(4)
print(f"Event-threshold metrics — Dataset {DATASET_NO}, Days {TARGET_DAYS}")
print(event_metrics_table.to_string(index=False))

summary_csv_path = GENERATED_PLOTS_DIR / f"event_threshold_metrics_dataset{DATASET_NO}.csv"
GENERATED_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
event_metrics_table.to_csv(summary_csv_path, index=False)
print(f"\nSaved event-threshold metrics table to: {summary_csv_path}")

event_metrics_table


### TEC-level summary (RMSE / MAE, TECU) — all forecast days

In [ ]:
tec_summary_rows = []
for day in TARGET_DAYS:
    r = results[day]
    m = tec_metrics[day]
    tec_summary_rows.append({
        "Day": day,
        "Date": r["date_label"] or f"day{day}",
        "LSTM RMSE (TECU)": m["lstm_rmse"],
        "LSTM MAE (TECU)": m["lstm_mae"],
        "Transformer RMSE (TECU)": m["trans_rmse"],
        "Transformer MAE (TECU)": m["trans_mae"],
    })

tec_summary_table = pd.DataFrame(tec_summary_rows).round(4)
print(tec_summary_table.to_string(index=False))

tec_summary_csv_path = GENERATED_PLOTS_DIR / f"tec_forecast_summary_dataset{DATASET_NO}.csv"
tec_summary_table.to_csv(tec_summary_csv_path, index=False)
print(f"\nSaved TEC forecast summary to: {tec_summary_csv_path}")

tec_summary_table
